# 09 Statistics and tables
Subject-level tests with Holm correction for the confirmatory family (H1 to H3), FDR for the rest, effect sizes with bootstrap intervals, summary tables. The figures are drawn in notebook 10.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
from retinopath import stats as S
nb = 1000 if DEV_MODE else cfg.stats.n_boot
ref = pd.read_csv(RES / "h1_reference.csv"); prot = pd.read_csv(RES / "protocols.csv"); lea = pd.read_csv(RES / "learned.csv")
lea["variant"] = lea.method.str.extract(r"net_(.*)_s\d+")[0]
clin = [p for p in cfg.clinical_protocols if p in cfg.run_protocols]; fam = []

# H1: RSI > 0 for each area pair and each retinotopic axis, reference data
r = ref[ref.method == "msmt_csd"]
for pair in ("V1-V2", "V2-V3", "V1-V3"):
    for q in ("dlogecc", "dang"):
        fam.append(dict(hyp="H1", test=f"RSI {q} {pair} > 0", **S.paired_test(r[f"rsi_{q}_{pair}"], alternative="greater", n_boot=nb)))

# H3: proposed (path loss) vs the same network with voxel loss only; seeds averaged within subject
for p in clin:
    w = lea[lea.protocol == p].pivot_table(index="sid", columns="variant", values=["rsi_ree_all", "acc"], aggfunc="mean")
    if {"path", "voxel"} <= set(w["rsi_ree_all"].columns):
        fam.append(dict(hyp="H3", test=f"RSI path > voxel-only, {p}", **S.paired_test(w["rsi_ree_all"]["path"], w["rsi_ree_all"]["voxel"], "greater", n_boot=nb)))
        fam.append(dict(hyp="H3", test=f"ACC path not worse than voxel-only, {p}", **S.paired_test(w["acc"]["path"], w["acc"]["voxel"], "greater", shift=cfg.stats.acc_margin, n_boot=nb)))

# H2: the voxel-only network recovers more of the lost voxel accuracy than of the lost RSI.
# recovery = (network - classical CSD on the same protocol) / (reference - classical CSD); reference ACC is 1 by definition.
lo = prot[prot.method == "csd"].set_index(["sid", "protocol"]); hi = r.set_index("sid")["rsi_ree_all"]
vx = lea[lea.variant == "voxel"].groupby(["sid", "protocol"])[["acc", "rsi_ree_all"]].mean()
j = vx.join(lo[["acc", "rsi_ree_all"]], rsuffix="_csd", how="inner").reset_index(); j["rsi_ref"] = j.sid.map(hi)
j["rec_acc"] = (j.acc - j.acc_csd) / (1 - j.acc_csd); lost = j.rsi_ref - j.rsi_ree_all_csd
j["rec_rsi"] = ((j.rsi_ree_all - j.rsi_ree_all_csd) / lost).where(lost >= 0.02)      # under 0.02 there is nothing to recover
j = j[j.protocol.isin(clin)]; print("subject x protocol cells without a loss of RSI (left out of H2):", int(j.rec_rsi.isna().sum()), "of", len(j))
rec = j.replace([np.inf, -np.inf], np.nan).groupby("sid")[["rec_acc", "rec_rsi"]].median()
fam.append(dict(hyp="H2", test="recovery of ACC > recovery of RSI, voxel-only network", **S.paired_test(rec.rec_acc, rec.rec_rsi, "greater", n_boot=nb)))

fam = pd.DataFrame(fam); fam["p_holm"] = S.correct(fam.p, "holm"); fam["significant"] = fam.p_holm < cfg.stats.alpha; fam.to_csv(RES / "confirmatory_tests.csv", index=False)
fam.round(4)

In [ ]:
# Exploratory grid: every method and protocol against classical CSD on the same protocol (FDR).
# Networks are averaged over training seeds first, so each variant is one method.
al = pd.concat([prot, lea.assign(method="net_" + lea.variant)]).groupby(["sid", "protocol", "method"], as_index=False)["rsi_ree_all"].mean(); rows = []
for (p, m), d in al.groupby(["protocol", "method"]):
    base = al[(al.protocol == p) & (al.method == "csd")].set_index("sid")["rsi_ree_all"]
    if m != "csd" and len(base):
        x = d.set_index("sid")["rsi_ree_all"].reindex(base.index)
        rows.append(dict(protocol=p, method=m, **S.paired_test(x, base, n_boot=nb)))
ex = pd.DataFrame(rows)
if len(ex): ex["p_fdr"] = S.correct(ex.p, "fdr_bh"); ex.to_csv(RES / "exploratory_tests.csv", index=False)
# Mixed model on the balanced part of the grid: clinical-grade protocols x the methods run on all of them
bal = al[al.protocol.isin(clin)]; full = bal.groupby("method").protocol.nunique(); bal = bal[bal.method.isin(full[full == len(clin)].index)]
mm = S.mixed_model(bal.rename(columns={"rsi_ree_all": "rsi"}))
if mm is not None: mm.to_csv(RES / "mixed_model.csv"); display(mm)
ex.round(4) if len(ex) else print("no exploratory comparisons")

In [ ]:
# Pairwise RSI comparisons between all methods on each clinical-grade protocol: two-sided Wilcoxon, FDR over every
# pair of every protocol (exploratory). Networks are averaged over training seeds first, so each variant is one method.
al = pd.concat([prot, lea.assign(method="net_" + lea.variant)])
al = al[al.protocol.isin(clin)].groupby(["sid", "protocol", "method"]).rsi_ree_all.mean().unstack("method"); rows = []
for p, d in al.groupby(level="protocol"):
    ms = [m for m in d.columns if d[m].notna().any()]
    for i, m1 in enumerate(ms):
        for m2 in ms[i + 1:]:
            rows.append(dict(protocol=p, method_1=m1, method_2=m2, **S.paired_test(d[m1], d[m2], n_boot=nb)))
pw = pd.DataFrame(rows)
if len(pw): pw["p_fdr"] = S.correct(pw.p, "fdr_bh"); pw.to_csv(RES / "pairwise_tests.csv", index=False)
print(len(pw), "pairs,", int((pw.p_fdr < cfg.stats.alpha).sum()) if len(pw) else 0, "with FDR p <", cfg.stats.alpha)

In [ ]:
# The path vs voxel-only contrast scored with the two keys that need no functional scan of the subject (descriptive)
alt = []
for p in clin:
    for col, key in (("template_rsi_ree_all", "group-average pRF"), ("atlas_rsi_ree_all", "anatomical atlas")):
        w = lea[lea.protocol == p].pivot_table(index="sid", columns="variant", values=col, aggfunc="mean")
        if {"path", "voxel"} <= set(w.columns):
            alt.append(dict(protocol=p, key=key, **S.paired_test(w["path"], w["voxel"], n_boot=nb)))
alt = pd.DataFrame(alt); alt.to_csv(RES / "alternative_key_tests.csv", index=False)
# agreement of the atlas key with the measured key over all subjects, protocols and methods
k = pd.concat([ref, prot, lea])[["rsi_ree_all", "atlas_rsi_ree_all"]].dropna()
print("Spearman r, measured-key RSI vs atlas-key RSI:", round(k.corr(method="spearman").iloc[0, 1], 3), "| n =", len(k)); alt.round(4)

In [ ]:
# Main table: median and inter-quartile range over subjects
fmt = lambda s: f"{s.median():.3f} [{s.quantile(.25):.3f}, {s.quantile(.75):.3f}]" if s.notna().any() else "not available"
al = pd.concat([ref[ref.method != "proximity"], prot, lea])
cols = {"rsi_ree_all": "RSI", "atlas_rsi_ree_all": "RSI, atlas key", "ree_median_all": "endpoint error (deg)", "acc": "ACC", "peak_err_deg": "peak error (deg)", "wdice_ref": "weighted Dice", "n_pairs": "V1-V3 streamlines"}
tab = al.groupby(["protocol", "method"])[list(cols)].agg(fmt).rename(columns=cols)
tab.to_csv(RES / "main_results.csv")
log = pd.read_csv(cfg.P.out / "run_log.csv"); print("hours logged so far:", round(log.hours.sum(), 2)); tab

In [ ]:
# External data: HCP 7T scans and NSD. Reported per method with medians; with 8 NSD subjects the paired comparison
# is shown with its interval and is descriptive, not part of the confirmatory family.
ext, rows = [], []
for name, f in (("HCP 7T", "external_7t.csv"), ("NSD", "external_nsd.csv")):
    if (RES / f).exists():
        d = pd.read_csv(RES / f); d["dataset"] = name; d["variant"] = d.method.str.extract(r"net_(.*)_s\d+")[0].fillna(d.method); ext.append(d)
        w = d.pivot_table(index="sid", columns="variant", values="rsi_ree_all", aggfunc="mean")
        if {"path", "voxel"} <= set(w.columns):
            rows.append(dict(dataset=name, test="RSI path - voxel-only", **S.paired_test(w["path"], w["voxel"], n_boot=nb)))
        if "csd" in w.columns and "path" in w.columns:
            rows.append(dict(dataset=name, test="RSI path - classical CSD", **S.paired_test(w["path"], w["csd"], n_boot=nb)))
if ext:
    ext = pd.concat(ext); cols2 = [c for c in ("rsi_ree_all", "rsi_ree_geodesic_all", "template_rsi_ree_all", "atlas_rsi_ree_all", "ree_median_all", "n_pairs") if c in ext]
    et = ext.groupby(["dataset", "variant"])[cols2].agg(fmt); et.to_csv(RES / "external_results.csv")
    pd.DataFrame(rows).to_csv(RES / "external_tests.csv", index=False); display(et); display(pd.DataFrame(rows).round(4))
rep = [pd.read_csv(RES / f).assign(dataset=n) for n, f in (("HCP retest", "retest_overlap.csv"), ("NSD run 1 vs 2", "nsd_run_overlap.csv")) if (RES / f).exists()]
if rep:
    rp = pd.concat(rep).groupby("dataset")[["wdice_retest", "badj_retest_mm", "vertex_count_r"]].agg(fmt); rp.to_csv(RES / "reproducibility.csv"); display(rp)